# Repository — in-memory users

Application code needs users, not SQL. `UserRepository` is a collection-like interface. `InMemoryUserRepository` stores a dict. A later SQL class could replace it without changing `display_name`.


In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


# --- Entity ---

@dataclass(frozen=True)
class User:
    id: str
    email: str
    name: str


In [2]:
# --- Repository ---

class UserRepository(ABC):
    @abstractmethod
    def add(self, user: User) -> None:
        ...

    @abstractmethod
    def get(self, user_id: str) -> User | None:
        ...

    @abstractmethod
    def list_all(self) -> list[User]:
        ...


In [3]:
# --- Concrete repository ---

class InMemoryUserRepository(UserRepository):
    def __init__(self) -> None:
        self._by_id: dict[str, User] = {}

    def add(self, user: User) -> None:
        self._by_id[user.id] = user

    def get(self, user_id: str) -> User | None:
        return self._by_id.get(user_id)

    def list_all(self) -> list[User]:
        return list(self._by_id.values())


In [4]:
# --- Client: depends on UserRepository, not on dict or SQL ---

def display_name(repo: UserRepository, user_id: str) -> str:
    user = repo.get(user_id)
    return 'unknown user' if user is None else user.name


repo = InMemoryUserRepository()
repo.add(User('u1', 'ada@example.com', 'Ada Lovelace'))
repo.add(User('u2', 'grace@example.com', 'Grace Hopper'))

print(display_name(repo, 'u1'))
print([user.email for user in repo.list_all()])
print(display_name(repo, 'missing'))


Ada Lovelace
['ada@example.com', 'grace@example.com']
unknown user


`display_name` never mentions the dict (or a database). The defining behavior of Repository is that the client treats storage as a collection of domain objects. Swap `InMemoryUserRepository` for a SQL implementation later; keep the client.
